# Esercizio 1

Carichiamo il dataset e facciamo un'analisi esplorativa

In [2]:
import pandas as pd

df = pd.read_csv("dataset_esercitazione.csv")

print(df.shape)
print(df.head())


(9105, 43)
        age     sex            dzgroup             dzclass  num.co   edu  \
0  62.84998    male        Lung Cancer              Cancer       0  11.0   
1  60.33899  female          Cirrhosis  COPD/CHF/Cirrhosis       2  12.0   
2  52.74698  female          Cirrhosis  COPD/CHF/Cirrhosis       2  12.0   
3  42.38498  female        Lung Cancer              Cancer       2  11.0   
4  79.88495  female  ARF/MOSF w/Sepsis            ARF/MOSF       1   NaN   

       income  scoma  charges  totcst  ...      crea    sod        ph  \
0    $11-$25k    0.0   9715.0     NaN  ...  1.199951  141.0  7.459961   
1    $11-$25k   44.0  34496.0     NaN  ...  5.500000  132.0  7.250000   
2  under $11k    0.0  41094.0     NaN  ...  2.000000  134.0  7.459961   
3  under $11k    0.0   3075.0     NaN  ...  0.799927  139.0       NaN   
4         NaN   26.0  50127.0     NaN  ...  0.799927  143.0  7.509766   

   glucose  bun  urine  adlp  adls  adlsc  death  
0      NaN  NaN    NaN   7.0   7.0    7.0 

Dobbiamo eliminare le variabili di **outcome**, in ambito medico e statistico, l'**outcome** (esito o risultato) è la conseguenza clinica finale osservata sul paziente (ad esempio: se il paziente è deceduto o meno, la durata della degenza, i costi totali dell'ospedalizzazione, o lo stato di salute a una certa scadenza).

Più in generale, scorrendo la tabella delle variabili nel PDF, rientrano tra gli outcome o le prognosi:
- le variabili sotto la categoria *costi*, $(charges, totcst, totmcst, avtisst)$.
- la variabile di esito principale/target come $death$ 

In [4]:
target = 'death'
cols_to_drop = ['charges', 'totcst', 'totmcst', 'avtisst', 'dzgroup', 'dzclass', target]

X = df.drop(columns=cols_to_drop)
y = df[target]

print(f"Dataset iniziale: {X.shape[1]}")
print(f"Target: {target}")
print(f"Eliminare: {cols_to_drop}")
print(X.head())


Dataset iniziale: 36
Target: death
Eliminare: ['charges', 'totcst', 'totmcst', 'avtisst', 'dzgroup', 'dzclass', 'death']
        age     sex  num.co   edu      income  scoma   race        sps   aps  \
0  62.84998    male       0  11.0    $11-$25k    0.0  other  33.898438  20.0   
1  60.33899  female       2  12.0    $11-$25k   44.0  white  52.695312  74.0   
2  52.74698  female       2  12.0  under $11k    0.0  white  20.500000  45.0   
3  42.38498  female       2  11.0  under $11k    0.0  white  20.097656  19.0   
4  79.88495  female       1   NaN         NaN   26.0  white  23.500000  30.0   

     surv2m  ...      bili      crea    sod        ph glucose  bun  urine  \
0  0.262939  ...  0.199982  1.199951  141.0  7.459961     NaN  NaN    NaN   
1  0.001000  ...       NaN  5.500000  132.0  7.250000     NaN  NaN    NaN   
2  0.790894  ...  2.199707  2.000000  134.0  7.459961     NaN  NaN    NaN   
3  0.698975  ...       NaN  0.799927  139.0       NaN     NaN  NaN    NaN   
4  0.634888  

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.05, random_state=42, stratify=y)

# Individuiamo il numero di features mancanti
print(f"Dati mancanti --> Train: {X_train.isnull().sum().sum()}, Test --> {X_test.isnull().sum().sum()} ")

# Individuiamo le colonne numeriche e categoriche
num_cols = X_train.select_dtypes(include=['float64', 'int64']).columns
cat_cols = X_train.select_dtypes(include=['string', 'object', 'category']).columns

# Creiamo la Pipeline di preprocessing

# 1. imputer con mediana e scaling per le features numeriche
num_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# 2. imputer con constant_value e OrdinalEncoder per le features categoriche
cat_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='Unkown')),
    ('encoder', OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

# impostiamo l'output in modo da non dover ricreare manualmente i DataFrame
preprocessor.set_output(transform='pandas')

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print(f"Dati mancanti --> Train: {X_train_processed.isnull().sum().sum()}, Test --> {X_test_processed.isnull().sum().sum()} ")


Dati mancanti --> Train: 38985, Test --> 2108 
Dati mancanti --> Train: 0, Test --> 0 
